# Notebook 3 - Exploratory Data Analysis (EDA)
Analyze the validated capital market dataset.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_parquet("processed/master_market_data.parquet")
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["Symbol","Date"])
df.head()


## Daily Returns

In [ ]:
df["Daily_Return"] = df.groupby("Symbol")["Close"].pct_change()
df[["Symbol","Date","Close","Daily_Return"]].head()


## Price Trend

In [ ]:
ticker="AAPL"   # Change as required
plot_df=df[df["Symbol"]==ticker]

plt.figure(figsize=(10,4))
plt.plot(plot_df["Date"],plot_df["Close"])
plt.title(f"{ticker} Closing Price Trend")
plt.xlabel("Date")
plt.ylabel("Close Price")
plt.grid(True)
plt.show()


## Volume Trend

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(plot_df["Date"],plot_df["Volume"])
plt.title(f"{ticker} Volume Trend")
plt.xlabel("Date")
plt.ylabel("Volume")
plt.grid(True)
plt.show()


## Top 10 Gainers

In [ ]:
latest=df.sort_values("Date").groupby("Symbol").tail(1)
top_gainers=latest.sort_values("Daily_Return",ascending=False).head(10)
top_gainers[["Symbol","Daily_Return"]]


## Top 10 Losers

In [ ]:
top_losers=latest.sort_values("Daily_Return").head(10)
top_losers[["Symbol","Daily_Return"]]


## Volatility

In [ ]:
volatility=df.groupby("Symbol")["Daily_Return"].std().reset_index()
volatility.columns=["Symbol","Volatility"]
volatility.sort_values("Volatility",ascending=False).head(10)


## ETF vs Stock

In [ ]:
comparison=df.groupby("Asset_Type").agg(
Average_Close=("Close","mean"),
Average_Volume=("Volume","mean"),
Average_Return=("Daily_Return","mean")
)
comparison


## Exchange Analysis

In [ ]:
if "Listing Exchange" in df.columns:
    exchange=df.groupby("Listing Exchange").agg(
        Instruments=("Symbol","nunique"),
        Avg_Close=("Close","mean"),
        Avg_Volume=("Volume","mean")
    )
    display(exchange)
else:
    print("Listing Exchange column not found.")


## Sector Analysis

In [ ]:
sector_col=None
for c in ["Sector","Industry","Market Category"]:
    if c in df.columns:
        sector_col=c
        break

if sector_col:
    sector=df.groupby(sector_col).agg(
        Instruments=("Symbol","nunique"),
        Avg_Close=("Close","mean"),
        Avg_Return=("Daily_Return","mean")
    )
    display(sector)
else:
    print("No sector-like column available.")


## Correlation

In [ ]:
corr=df[["Open","High","Low","Close","Adj Close","Volume"]].corr()
corr


## Save Summary

In [ ]:
latest.to_csv("processed/latest_prices.csv",index=False)
volatility.to_csv("processed/volatility_summary.csv",index=False)
print("EDA summaries saved.")
